# 02 — Train Mental-Health Classifiers (Low et al. methodology)

Trains three binary LinearSVC classifiers following Low et al. (2020) on mental health subreddit posts, then saves the fitted pipelines for use in notebooks 03 and 04.

**Training data** (Jan 2022–Jul 2023, before the study window):
- Positive: r/anxiety, r/depression, r/stress — 2,000 posts each
- Negative: r/personalfinance, r/learnprogramming, r/todayilearned, r/careerguidance — 2,000 each

**Features**: TF-IDF unigrams + bigrams, max 50k features, validated via 5-fold CV (F1: 0.88–0.94).

**Outputs**: `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib`

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import json
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.utils import shuffle

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed'
RAW_DIR  = ROOT / 'data' / 'raw'
FIG_DIR  = ROOT / 'figures'
MODEL_DIR = ROOT / 'models'

BASE_URL    = 'https://arctic-shift.photon-reddit.com'
RATE_LIMIT  = 0.4   # seconds between requests

# Training window: before our study period to avoid data leakage
TRAIN_AFTER  = '2022-01-01'
TRAIN_BEFORE = '2023-07-31'

# Posts per subreddit to fetch (balanced classes)
N_PER_SUB = 2000

TRAIN_DATA_PATH = DATA_DIR / 'training_data_raw.parquet'
print('Setup complete')

Setup complete


## 1) Pull training data from Arctic Shift

In [2]:
# Positive subreddits (mental health) — same as Low et al. 2020
MH_SUBS = ['anxiety', 'depression', 'stress']

# Control subreddits: avoid very large subs (r/AskReddit times out per API docs)
CTRL_SUBS = ['personalfinance', 'learnprogramming', 'todayilearned', 'careerguidance']

session = requests.Session()
session.headers.update({'User-Agent': 'CS598-research/1.0 (uiuc; gradadmissions study)'})

def fetch_posts_for_sub(subreddit, n=2000, after=TRAIN_AFTER, before=TRAIN_BEFORE):
    """Paginate Arctic Shift /api/posts/search to collect n posts from a subreddit.

    API docs: https://github.com/ArthurHeitmann/arctic_shift/blob/master/api/README.md
    Endpoint: /api/posts/search
    Pagination: use created_utc of last record as next `after` value (epoch seconds accepted).
    """
    posts = []
    after_ts = after
    while len(posts) < n:
        params = {
            'subreddit': subreddit,
            'after':     after_ts,
            'before':    before,
            'limit':     'auto',   # returns 100-1000 depending on server capacity
            'sort':      'asc',    # sort by created_utc ascending
        }
        try:
            r = session.get(f'{BASE_URL}/api/posts/search', params=params, timeout=30)
            rl_remaining = r.headers.get('X-RateLimit-Remaining')

            if r.status_code == 429:
                rl_reset = r.headers.get('X-RateLimit-Reset')
                wait = 10
                if rl_reset:
                    try: wait = max(1, float(rl_reset) - time.time()) + 1
                    except: pass
                print(f'    Rate limited — waiting {wait:.0f}s')
                time.sleep(wait)
                continue

            if r.status_code != 200:
                print(f'    HTTP {r.status_code} for r/{subreddit}: {r.text[:300]}')
                break

            data = r.json().get('data', [])
            if not data:
                break

            for p in data:
                text = ' '.join(filter(None, [p.get('title', ''), p.get('selftext', '')])).strip()
                if len(text) > 20 and text not in ('[deleted]', '[removed]'):
                    posts.append({
                        'text':        text,
                        'subreddit':   subreddit,
                        'created_utc': p.get('created_utc', ''),
                    })

            # Pagination: advance `after` to created_utc of last record (epoch seconds)
            last_ts = data[-1].get('created_utc', '')
            if not last_ts or last_ts == after_ts:
                break   # no new data
            after_ts = last_ts

            # Dynamic rate limiting per X-RateLimit-Remaining header
            if rl_remaining:
                try:
                    rem = int(rl_remaining)
                    if rem < 5:    time.sleep(2.0)
                    elif rem < 20: time.sleep(0.8)
                    else:          time.sleep(RATE_LIMIT)
                except:
                    time.sleep(RATE_LIMIT)
            else:
                time.sleep(RATE_LIMIT)

        except Exception as e:
            print(f'    Error: {e}')
            time.sleep(2)
            break

    return posts[:n]


if TRAIN_DATA_PATH.exists():
    print(f'Training data already cached at {TRAIN_DATA_PATH}')
    train_raw = pd.read_parquet(TRAIN_DATA_PATH)
    print(train_raw['subreddit'].value_counts())
else:
    all_records = []
    for sub in MH_SUBS + CTRL_SUBS:
        print(f'Fetching r/{sub}...')
        recs = fetch_posts_for_sub(sub, n=N_PER_SUB)
        all_records.extend(recs)
        print(f'  → {len(recs):,} posts collected')

    train_raw = pd.DataFrame(all_records)
    if len(train_raw) > 0:
        train_raw.to_parquet(TRAIN_DATA_PATH, index=False)
        print(f'\nSaved {len(train_raw):,} training records')
        print(train_raw['subreddit'].value_counts())
    else:
        print('No data collected — check API response above')

Training data already cached at /media/ayush/F/Coding/CS598_Research_Project/data/processed/training_data_raw.parquet
subreddit
anxiety             2000
depression          2000
stress              2000
personalfinance     2000
learnprogramming    2000
todayilearned       2000
careerguidance      2000
Name: count, dtype: int64


## 2) Train classifiers (one per mental-health class)

In [3]:
# Build combined control pool (all control posts)
ctrl_df = train_raw[train_raw['subreddit'].isin(CTRL_SUBS)].copy()
print(f'Control pool: {len(ctrl_df):,} posts')

classifiers = {}  # will hold {name: fitted_pipeline}

for mh_sub in MH_SUBS:
    print(f'\n--- Training {mh_sub} classifier ---')
    pos_df = train_raw[train_raw['subreddit'] == mh_sub].copy()
    
    # Sample equal number of controls (balanced classes)
    n_pos = len(pos_df)
    neg_df = ctrl_df.sample(n=min(n_pos, len(ctrl_df)), random_state=42)
    
    pos_df['label'] = 1
    neg_df['label'] = 0
    
    combined = pd.concat([pos_df, neg_df], ignore_index=True)
    combined = shuffle(combined, random_state=42)
    
    X = combined['text'].tolist()
    y = combined['label'].values
    
    print(f'  Positive (r/{mh_sub}): {n_pos:,}  |  Negative (control): {len(neg_df):,}')
    
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=50_000,
            sublinear_tf=True,
            min_df=3,
            strip_accents='unicode',
            analyzer='word',
        )),
        ('clf', LinearSVC(C=1.0, max_iter=2000, class_weight='balanced')),
    ])
    
    # 5-fold cross-validation
    cv_scores = cross_val_score(pipe, X, y, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1)
    print(f'  CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
    
    # Fit on full training set
    pipe.fit(X, y)
    classifiers[mh_sub] = pipe
    
    # Save model
    model_path = MODEL_DIR / f'clf_{mh_sub}.joblib'
    joblib.dump(pipe, model_path)
    print(f'  Saved → {model_path.name}')

print('\nAll classifiers trained.')

Control pool: 8,000 posts

--- Training anxiety classifier ---
  Positive (r/anxiety): 2,000  |  Negative (control): 2,000
  CV F1: 0.883 ± 0.010
  Saved → clf_anxiety.joblib

--- Training depression classifier ---
  Positive (r/depression): 2,000  |  Negative (control): 2,000
  CV F1: 0.884 ± 0.009
  Saved → clf_depression.joblib

--- Training stress classifier ---
  Positive (r/stress): 2,000  |  Negative (control): 2,000
  CV F1: 0.938 ± 0.006
  Saved → clf_stress.joblib

All classifiers trained.


## 3) Quick evaluation on held-out test sample

In [4]:
from sklearn.model_selection import train_test_split

for mh_sub in MH_SUBS:
    pipe = classifiers[mh_sub]
    pos_df = train_raw[train_raw['subreddit'] == mh_sub].copy()
    neg_df = ctrl_df.sample(n=len(pos_df), random_state=42)
    pos_df['label'] = 1; neg_df['label'] = 0
    combined = shuffle(pd.concat([pos_df, neg_df], ignore_index=True), random_state=99)
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        combined['text'].tolist(), combined['label'].values,
        test_size=0.2, random_state=99, stratify=combined['label'].values
    )
    pipe_eval = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50_000,
                                  sublinear_tf=True, min_df=3)),
        ('clf', LinearSVC(C=1.0, max_iter=2000, class_weight='balanced')),
    ])
    pipe_eval.fit(X_tr, y_tr)
    y_pred = pipe_eval.predict(X_te)
    print(f'\n=== {mh_sub} (80/20 split) ===')
    print(classification_report(y_te, y_pred, target_names=['control', mh_sub]))


=== anxiety (80/20 split) ===
              precision    recall  f1-score   support

     control       0.94      0.76      0.84       400
     anxiety       0.80      0.95      0.87       400

    accuracy                           0.86       800
   macro avg       0.87      0.86      0.85       800
weighted avg       0.87      0.86      0.85       800


=== depression (80/20 split) ===
              precision    recall  f1-score   support

     control       0.95      0.75      0.84       400
  depression       0.79      0.96      0.87       400

    accuracy                           0.85       800
   macro avg       0.87      0.85      0.85       800
weighted avg       0.87      0.85      0.85       800


=== stress (80/20 split) ===
              precision    recall  f1-score   support

     control       0.94      0.95      0.95       400
      stress       0.95      0.94      0.95       400

    accuracy                           0.95       800
   macro avg       0.95      0.95